# MLX Model Testing — Llama & Qwen on Apple Silicon
## Strategy: GGUF/Ollama → Native MLX for Mac M4 16 GB

<table style="margin: 0; text-align: left; width:100%">
<tr>
  <td style="width: 250px; height: 150px; vertical-align: middle;">
    <img src="assets/logo.png" width="250" height="150" style="display: block;" />
  </td>
  <td>
    <h2 style="color:#ff7800;">Objectives</h2>
    <span style="color:#ff7800;">
      - Remove Ollama GGUF models and free ~6.8 GB of storage<br>
      - Install <code>mlx-lm</code> for Apple Silicon native inference<br>
      - Load 4-bit MLX builds of Llama 3.2 3B and Qwen3 8B<br>
      - Test code generation and chat completions side-by-side<br>
      - Observe ~1.5–2× faster tokens/s vs GGUF on M4
    </span>
  </td>
</tr>
</table>

## Migration Overview

| | Before (Ollama GGUF) | After (MLX native) |
|---|---|---|
| **Llama model** | `llama3.2:latest` 3.2B Q4\_K\_M | `mlx-community/Llama-3.2-3B-Instruct-4bit` |
| **Qwen model** | `qwen3:8b` 8.2B Q4\_K\_M | `mlx-community/Qwen3-8B-4bit` |
| **Llama disk** | 1.9 GB (Ollama blobs) | ~1.8 GB (HF cache) |
| **Qwen disk** | 4.9 GB (Ollama blobs) | ~4.5 GB (HF cache) |
| **Runtime** | llama.cpp via Ollama server | Apple MLX Metal + Neural Engine |
| **Speed gain** | baseline | ~1.5–2× faster tok/s on M4 |
| **API** | OpenAI-compatible REST | Direct Python — no server needed |

> **Important:** Load and unload one model at a time to stay within the 16 GB unified memory budget.
> Run cells **in order** top to bottom.

In [1]:
# Step 1 — Remove existing GGUF models to reclaim ~6.8 GB
# Requires the Ollama server to be running: `ollama serve`
import subprocess
import pathlib

models_to_remove = ["llama3.2", "qwen3:8b"]

for model in models_to_remove:
    result = subprocess.run(
        ["ollama", "rm", model],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"Removed: {model}")
    else:
        msg = (result.stderr or result.stdout).strip()
        print(f"Skipped {model}: {msg or 'already removed'}")

ollama_dir = pathlib.Path.home() / ".ollama" / "models"
size = subprocess.run(["du", "-sh", str(ollama_dir)],
                      capture_output=True, text=True)
print(f"\nOllama models dir remaining: {size.stdout.strip()}")

Skipped llama3.2: Error: could not connect to ollama server, run 'ollama serve' to start it
Skipped qwen3:8b: Error: could not connect to ollama server, run 'ollama serve' to start it

Ollama models dir remaining: 0B	/Users/abhisheksaxena/.ollama/models


In [ ]:
# Step 2 — Install mlx-lm (Apple Silicon only)
import platform, subprocess, sys

if platform.machine() != "arm64":
    print("ERROR: MLX requires Apple Silicon (arm64). Not supported on Intel Macs.")
else:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "mlx-lm"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("Install error:", result.stderr[-400:])
    else:
        print("mlx-lm installed successfully.")
        print("If this was a first-time install, restart the kernel before continuing.")

In [2]:
# Common imports — run after restarting kernel if mlx-lm was just installed
import time
import mlx.core as mx
from mlx_lm import load, generate
from IPython.display import Markdown, display

import mlx_lm
print(f"mlx-lm version : {mlx_lm.__version__}")
print(f"MLX metal device: {mx.default_device()}")

/Users/abhisheksaxena/Tutorials/Agentic-AI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mlx-lm version : 0.31.2
MLX metal device: Device(gpu, 0)


---

## Part 1 — Llama 3.2 3B Instruct (MLX 4-bit)

| Property | Value |
|---|---|
| **Model** | `mlx-community/Llama-3.2-3B-Instruct-4bit` |
| **Parameters** | 3.2B |
| **Disk size** | ~1.8 GB (downloaded to `~/.cache/huggingface/hub/`) |
| **Context** | 128k tokens |
| **Strengths** | Fast responses, instruction following, code generation |

> First run downloads ~1.8 GB automatically. Subsequent runs load from local cache.

In [ ]:
# Load Llama 3.2 3B Instruct — first run downloads ~1.8 GB from HuggingFace
LLAMA_MODEL = "mlx-community/Llama-3.2-3B-Instruct-4bit"

print(f"Loading {LLAMA_MODEL} ...")
t0 = time.time()
llama_model, llama_tokenizer = load(LLAMA_MODEL)
print(f"Model loaded in {time.time() - t0:.1f}s")

In [ ]:
# Test 1 — Code generation with streaming output (verbose=True streams tokens live)
prompt_text = "Write a short Python function that reverses a string."
messages = [{"role": "user", "content": prompt_text}]

llama_prompt = llama_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print(f"Prompt: {prompt_text}")
print("-" * 60)
t0 = time.time()
llama_stream_response = generate(
    llama_model, llama_tokenizer,
    prompt=llama_prompt,
    max_tokens=512,
    verbose=True  # streams tokens to stdout as they are generated
)
elapsed = time.time() - t0
n_tokens = len(llama_tokenizer.encode(llama_stream_response))
print(f"\n" + "-" * 60)
print(f"Generated {n_tokens} tokens in {elapsed:.1f}s  ->  {n_tokens / elapsed:.1f} tok/s")

In [ ]:
# Test 2 — Chat completion (system + user message), markdown table output
chat_request = (
    "Compare the population of New York and Los Angeles. "
    "Provide the answer in a markdown table format."
)
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user",   "content": chat_request},
]
llama_chat_prompt = llama_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

t0 = time.time()
llama_answer = generate(
    llama_model, llama_tokenizer,
    prompt=llama_chat_prompt,
    max_tokens=400,
    verbose=False
)
elapsed = time.time() - t0
n_tokens = len(llama_tokenizer.encode(llama_answer))
print(f"Generated {n_tokens} tokens in {elapsed:.1f}s  ->  {n_tokens / elapsed:.1f} tok/s")
print(llama_answer)

In [ ]:
# Render the chat answer as formatted Markdown
display(Markdown(llama_answer))

In [ ]:
# Unload Llama and clear Metal GPU cache before loading Qwen
del llama_model, llama_tokenizer
mx.metal.clear_cache()
print("Llama model unloaded. Metal cache cleared. Ready to load Qwen.")

---

## Part 2 — Qwen3 8B (MLX 4-bit)

| Property | Value |
|---|---|
| **Model** | `mlx-community/Qwen3-8B-4bit` |
| **Parameters** | 8.2B |
| **Disk size** | ~4.5 GB (stored in `~/AIModels/Qwen3-8B-4bit/`) |
| **Context** | 128k tokens |
| **Strengths** | Complex reasoning, multilingual, tool calling, structured output |

### Thinking Mode
Qwen3 supports a reasoning mode that produces a `<think>...</think>` block before answering.

| `enable_thinking` | Use case |
|---|---|
| `False` | Fast everyday answers, chat, code completion |
| `True` | Hard math, multi-step logic, complex coding problems |

> First run downloads ~4.5 GB into the shared AIModels folder. Subsequent runs reuse the same local copy.

In [3]:
# Load Qwen3 8B from a shared local model folder under ~/AIModels
from pathlib import Path
from huggingface_hub import snapshot_download

QWEN_MODEL = "mlx-community/Qwen3-8B-4bit"
AI_MODELS_DIR = Path.home() / "AIModels"
QWEN_LOCAL_DIR = AI_MODELS_DIR / "Qwen3-8B-4bit"

QWEN_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Shared model location: {QWEN_LOCAL_DIR}")

# Download once into the shared folder, then always reuse it
snapshot_download(
    repo_id=QWEN_MODEL,
    local_dir=str(QWEN_LOCAL_DIR),
    local_dir_use_symlinks=False,
)

print(f"Loading {QWEN_MODEL} from {QWEN_LOCAL_DIR} ...")
t0 = time.time()
qwen_model, qwen_tokenizer = load(str(QWEN_LOCAL_DIR))
print(f"Model loaded in {time.time() - t0:.1f}s")

/Users/abhisheksaxena/Tutorials/Agentic-AI/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Shared model location: /Users/abhisheksaxena/AIModels/Qwen3-8B-4bit


Fetching 11 files: 100%|██████████| 11/11 [00:41<00:00,  3.79s/it]


Loading mlx-community/Qwen3-8B-4bit from /Users/abhisheksaxena/AIModels/Qwen3-8B-4bit ...
Model loaded in 17.1s


In [4]:
# Test 1 — Code generation with streaming output, thinking mode OFF
prompt_text = "Write a short Python function that reverses a string."
messages = [{"role": "user", "content": prompt_text}]

# Qwen3 chat template supports enable_thinking parameter
try:
    qwen_prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False  # set True for step-by-step reasoning
    )
except TypeError:
    # Fallback for tokenizer versions without enable_thinking
    qwen_prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print(f"Prompt: {prompt_text}")
print("-" * 60)
t0 = time.time()
qwen_stream_response = generate(
    qwen_model, qwen_tokenizer,
    prompt=qwen_prompt,
    max_tokens=512,
    verbose=True  # streams tokens to stdout as they are generated
)
elapsed = time.time() - t0
n_tokens = len(qwen_tokenizer.encode(qwen_stream_response))
print(f"\n" + "-" * 60)
print(f"Generated {n_tokens} tokens in {elapsed:.1f}s  ->  {n_tokens / elapsed:.1f} tok/s")

Prompt: Write a short Python function that reverses a string.
------------------------------------------------------------
Sure! Here's a short Python function that reverses a string:

```python
def reverse_string(s):
    return s[::-1]
```

### Example usage:
```python
original = "hello"
reversed_str = reverse_string(original)
print(reversed_str)  # Output: "olleh"
```

This uses Python's slicing syntax `s[::-1]` to reverse the string.
Prompt: 23 tokens, 10.177 tokens-per-sec
Generation: 83 tokens, 20.379 tokens-per-sec
Peak memory: 4.687 GB

------------------------------------------------------------
Generated 82 tokens in 6.5s  ->  12.6 tok/s


In [ ]:
# Test 2 — Chat completion (system + user message), markdown table output
chat_request = (
    "Compare the population of New York and Los Angeles. "
    "Provide the answer in a markdown table format."
)
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user",   "content": chat_request},
]

try:
    qwen_chat_prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
except TypeError:
    qwen_chat_prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

t0 = time.time()
qwen_answer = generate(
    qwen_model, qwen_tokenizer,
    prompt=qwen_chat_prompt,
    max_tokens=400,
    verbose=False
)
elapsed = time.time() - t0
n_tokens = len(qwen_tokenizer.encode(qwen_answer))
print(f"Generated {n_tokens} tokens in {elapsed:.1f}s  ->  {n_tokens / elapsed:.1f} tok/s")
print(qwen_answer)

In [ ]:
# Render the chat answer as formatted Markdown
display(Markdown(qwen_answer))

In [ ]:
# Unload Qwen and clear Metal cache when finished
del qwen_model, qwen_tokenizer
mx.metal.clear_cache()
print("Qwen model unloaded. Metal cache cleared.")

---

## Tips & Next Steps

### Memory Budget (16 GB M4)
| Component | Size |
|---|---|
| macOS + system | ~4–5 GB |
| Llama 3.2 3B 4-bit | ~2 GB |
| Qwen3 8B 4-bit | ~5 GB |
| **Cannot run both at once** | **always unload before loading the other** |

### Model Cache Location
MLX models are stored in `~/.cache/huggingface/hub/`.  
To remove a model manually:
```bash
rm -rf ~/.cache/huggingface/hub/models--mlx-community--Qwen3-8B-4bit
rm -rf ~/.cache/huggingface/hub/models--mlx-community--Llama-3.2-3B-Instruct-4bit
```

### Run MLX as an OpenAI-compatible Server
If you want to use your existing `ollama_chat.py` or notebooks that call the OpenAI API, you can serve MLX models as a drop-in Ollama replacement:
```bash
mlx_lm.server --model mlx-community/Qwen3-8B-4bit --port 8080
```
Then update the base URL in your code:
```python
client = OpenAI(base_url="http://localhost:8080/v1", api_key="mlx")
```

### Enable Thinking Mode (Qwen3 only)
Change `enable_thinking=False` to `True` in the `apply_chat_template` calls above.  
Expect a `<think>...</think>` reasoning block in the output and longer generation times.